# Business Impact Analysis

Objective:
- Estimate revenue loss due to theft
- Evaluate model-driven inspection strategy
- Calculate ROI
- Quantify business value

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

In [ ]:
df = pd.read_csv("../data/processed/final_dataset_with_graph.csv")

df.head()

In [ ]:
import joblib

model = joblib.load("../models/classification/xgb_model.pkl")

model

In [ ]:
feature_columns = [
    "kwh",
    "voltage",
    "current",
    "power_factor",
    "temperature",
    "humidity",
    "hour",
    "day",
    "month",
    "day_of_week",
    "is_weekend",
    "is_night",
    "is_peak_hour",
    "load_ratio",
    "rolling_mean_24h",
    "rolling_std_24h",
    "lag_1",
    "lag_24",
    "diff_1",
    "diff_24",
    "z_score",
    "transformer_deviation",
    "rolling_max_24h",
    "rolling_min_24h",
    "sudden_drop_flag",
    "isolation_score"
]

X = df[feature_columns]
y = df["theft_label"]

In [ ]:
df["theft_probability"] = model.predict_proba(X)[:, 1]

In [ ]:
INSPECTION_COST = 2000        # ₹ per inspection
AVERAGE_MONTHLY_BILL = 1500   # ₹ per consumer
THEFT_LOSS_MULTIPLIER = 3     # theft consumers underpay by 3x

In [ ]:
theft_cases = df[df["theft_label"] == 1]

estimated_loss = len(theft_cases) * AVERAGE_MONTHLY_BILL * THEFT_LOSS_MULTIPLIER

print("Estimated Monthly Revenue Loss (₹):", estimated_loss)

In [ ]:
threshold = df["theft_probability"].quantile(0.95)

df["inspection_flag"] = (df["theft_probability"] >= threshold).astype(int)

print("Customers selected for inspection:", df["inspection_flag"].sum())

In [ ]:
cm = confusion_matrix(df["theft_label"], df["inspection_flag"])

tn, fp, fn, tp = cm.ravel()

print("True Positives:", tp)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Negatives:", tn)

In [ ]:
recovered_revenue = tp * AVERAGE_MONTHLY_BILL * THEFT_LOSS_MULTIPLIER

inspection_cost_total = (tp + fp) * INSPECTION_COST

net_profit = recovered_revenue - inspection_cost_total

print("Recovered Revenue (₹):", recovered_revenue)
print("Inspection Cost (₹):", inspection_cost_total)
print("Net Business Impact (₹):", net_profit)

In [ ]:
roi = (net_profit / inspection_cost_total) * 100

print("Return on Investment (%):", roi)

In [ ]:
random_sample = df.sample(df["inspection_flag"].sum(), random_state=42)

random_tp = len(random_sample[random_sample["theft_label"] == 1])

random_recovery = random_tp * AVERAGE_MONTHLY_BILL * THEFT_LOSS_MULTIPLIER
random_cost = len(random_sample) * INSPECTION_COST

random_profit = random_recovery - random_cost

print("Random Strategy Net Profit (₹):", random_profit)

In [ ]:
strategies = ["Model-Based", "Random"]
profits = [net_profit, random_profit]

plt.bar(strategies, profits)
plt.title("Profit Comparison")
plt.ylabel("Net Profit (₹)")
plt.show()

## Business Impact Summary

1. AI-based inspection significantly improves revenue recovery.
2. Risk-based strategy reduces unnecessary inspections.
3. ROI is positive compared to random strategy.
4. Fraud detection reduces Non-Technical Losses (NTL).
5. Model enables data-driven operational decisions.

This system:
- Improves inspection efficiency
- Reduces revenue leakage
- Enhances grid transparency